In [1]:
import tensorflow as tf
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.preprocessing import LabelEncoder
from pathlib import Path
import glob
import json
import pickle
import matplotlib.pyplot as plt
!pip install rdkit --break-system-packages
from rdkit import Chem
from rdkit.Chem import MolFromSmarts
from google.colab import drive
drive.mount('/content/drive')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 66.4 MB/s eta 0:00:00
Mounted at /content/drive


In [2]:
def crpic(points, positions, hauteur, largeur=0.002):
    return hauteur * (largeur**2) / ((points - positions)**2 + largeur**2)

def normaliser_intensites(df):
    max_int = df['Int.'].max()
    df['Int.'] = df['Int.'] / max_int * 1000
    return df

points_1H = np.linspace(0, 12, 30000)

In [3]:
SOLVANTS_1H = {
    'cdcl3':   {'ppm': 7.26,  'Int.': 500},
    'ccl4':    None,
    'dmso-d6': {'ppm': 2.50,  'Int.': 700},
    'd2o':     {'ppm': 4.75,  'Int.': 600},
}

# Molecules 1H
molecules_1H = {}
for dossier in glob.glob('/content/drive/MyDrive/IA RMN/Molecules CSV 1H/*/'):
    nom_solvant = Path(dossier).name.lower()
    pic_solvant = SOLVANTS_1H.get(nom_solvant, None)
    for fichier in glob.glob(f'{dossier}*.csv'):
        nom = Path(fichier).stem.split('(')[0]
        try:
            df = pd.read_csv(fichier)[['ppm', 'Int.']]
            df['ppm'] = pd.to_numeric(df['ppm'], errors='coerce')
            df['Int.'] = pd.to_numeric(df['Int.'], errors='coerce')
            df.dropna(inplace=True)
            df = normaliser_intensites(df)
            molecules_1H[nom] = {'df': df, 'solvant': pic_solvant}
        except Exception as e:
            print(f"Erreur {fichier}: {e}")

# Annotations
df_annotations = pd.read_csv('/content/drive/MyDrive/IA RMN/mon_modele_tf 1H/annotations_fonctions.csv',
                              index_col='molecule')

print(f"Molécules 1H       : {len(molecules_1H)}")
print(f"Molécules annotées : {len(df_annotations)}")

Molécules 1H       : 1385
Molécules annotées : 1231


In [4]:
# Weight and list of functions
with open('/content/drive/MyDrive/IA RMN/mon_modele_tf 1H/pos_weights.json') as f:
    poids_data = json.load(f)
colonnes_fonctions = poids_data['colonnes']
n_fonctions = len(colonnes_fonctions)

freq = df_annotations[colonnes_fonctions].mean().values
pos_weights = np.sqrt((1 - freq) / freq).astype(np.float32)

print(f"Molécules 1H       : {len(molecules_1H)}")
print(f"Molécules annotées : {len(df_annotations)}")
print(f"Fonctions          : {n_fonctions} → {colonnes_fonctions}")
print("\nPoids adoucis :")
for f, w in zip(colonnes_fonctions, pos_weights):
    print(f"  {f:<22} {w:.2f}")

Molécules 1H       : 1385
Molécules annotées : 1231
Fonctions          : 15 → ['aromatique', 'alcool', 'phenol', 'acide_carboxylique', 'cetone', 'aldehyde', 'amine', 'ester', 'ether', 'halogenure', 'nitrile', 'amide', 'heterocycle_n', 'sulfoxyde', 'alcene']

Poids adoucis :
  aromatique             1.12
  alcool                 1.60
  phenol                 2.84
  acide_carboxylique     1.58
  cetone                 2.32
  aldehyde               2.77
  amine                  1.75
  ester                  2.49
  ether                  2.53
  halogenure             2.63
  nitrile                3.34
  amide                  2.22
  heterocycle_n          1.98
  sulfoxyde              4.54
  alcene                 2.49


In [5]:
class SpectreGeneratorMultiLabel(tf.keras.utils.Sequence):
    def __init__(self, molecules_dict, df_annotations, colonnes_fonctions,
                 points, batch_size=128, n_per_molecule=50, augmentation=True,
                 workers=4, use_multiprocessing=True, max_queue_size=20, **kwargs):
        super().__init__(workers=workers, use_multiprocessing=use_multiprocessing,
                         max_queue_size=max_queue_size, **kwargs)
        self.molecules = [m for m in molecules_dict.keys() if m in df_annotations.index]
        self.points = points
        self.batch_size = batch_size
        self.augmentation = augmentation
        self.total = len(self.molecules) * n_per_molecule
        self.annotations = df_annotations[colonnes_fonctions]
        self.arrays = {m: (molecules_dict[m]['df']['ppm'].values,
                           molecules_dict[m]['df']['Int.'].values,
                           molecules_dict[m]['solvant']) for m in self.molecules}
        self.labels = {m: self.annotations.loc[m].values.astype(np.float32)
                       for m in self.molecules}
        print(f"Molécules avec annotations : {len(self.molecules)}")

    def __len__(self):
        return self.total // self.batch_size

    def __getitem__(self, idx):
        X = np.empty((self.batch_size, len(self.points), 1), dtype=np.float32)
        Y = np.empty((self.batch_size, len(self.annotations.columns)), dtype=np.float32)

        for i in range(self.batch_size):
            nom = np.random.choice(self.molecules)
            ppm_arr, int_arr, pic_solvant = self.arrays[nom]

            if self.augmentation:
                X_rand = np.random.uniform(-0.1, 0.1)
                Y_rand = np.random.uniform(0.8, 1.2)
                largeur = np.random.uniform(0.0015, 0.004)
            else:
                X_rand, Y_rand, largeur = 0.0, 1.0, 0.002

            spectre = np.zeros_like(self.points)
            for p, h in zip(ppm_arr, int_arr):
                spectre += crpic(self.points, p + X_rand, h * Y_rand, largeur)

            if pic_solvant is not None and self.augmentation and np.random.random() < 0.7:
                spectre += crpic(self.points,
                                 pic_solvant['ppm'] + np.random.uniform(-0.02, 0.02),
                                 pic_solvant['Int.'] * np.random.uniform(0.8, 1.2),
                                 np.random.uniform(0.0015, 0.004))

            if self.augmentation:
                amax = np.max(spectre)
                snr = 10 ** np.random.uniform(np.log10(150), np.log10(500))
                spectre += np.random.normal(0, amax / snr if amax > 0 else 0.01, len(self.points))

            X[i, :, 0] = spectre
            Y[i] = self.labels[nom]

        return X, Y

In [6]:
import gc

np.random.seed(42)
gen_val_temp = SpectreGeneratorMultiLabel(
    molecules_1H, df_annotations, colonnes_fonctions,
    points_1H, batch_size=128, n_per_molecule=10, augmentation=True,
    workers=1, use_multiprocessing=False
)
X_val_list, Y_val_list = [], []
for i in range(len(gen_val_temp)):
    xb, yb = gen_val_temp[i]
    X_val_list.append(xb); Y_val_list.append(yb)
X_val_ml = np.concatenate(X_val_list, axis=0)
Y_val_ml = np.concatenate(Y_val_list, axis=0)
del X_val_list, Y_val_list, gen_val_temp; gc.collect()
print(f"Validation : {X_val_ml.shape}, labels : {Y_val_ml.shape}")

Molécules avec annotations : 1231
Validation : (12288, 30000, 1), labels : (12288, 15)


In [7]:
model_1H = tf.keras.models.load_model(
    '/content/drive/MyDrive/IA RMN/mon_modele_tf 1H/5_modele_RMN1H.h5'
)

dummy = tf.zeros((1, 30000, 1))
_ = model_1H(dummy)

for i, layer in enumerate(model_1H.layers):
    try:
        shape = layer.output.shape
    except Exception:
        shape = "?"
    print(f"  [{i}]  {layer.name:<25} {type(layer).__name__:<20} → {shape}")

  [0]  conv1d_9                  Conv1D               → (None, 7496, 32)
  [1]  batch_normalization_9     BatchNormalization   → (None, 7496, 32)
  [2]  max_pooling1d_9           MaxPooling1D         → (None, 1874, 32)
  [3]  conv1d_10                 Conv1D               → (None, 1870, 64)
  [4]  batch_normalization_10    BatchNormalization   → (None, 1870, 64)
  [5]  max_pooling1d_10          MaxPooling1D         → (None, 467, 64)
  [6]  conv1d_11                 Conv1D               → (None, 463, 128)
  [7]  batch_normalization_11    BatchNormalization   → (None, 463, 128)
  [8]  max_pooling1d_11          MaxPooling1D         → (None, 115, 128)
  [9]  average_pooling1d_3       AveragePooling1D     → (None, 14, 128)
  [10]  flatten_3                 Flatten              → (None, 1792)
  [11]  dense_6                   Dense                → (None, 64)
  [12]  dropout_3                 Dropout              → (None, 64)
  [13]  dense_7                   Dense                → (None, 13

In [8]:
k = 11   # Index of the Dense(64) layer, found from the summary above

feature_extractor = tf.keras.Sequential(model_1H.layers[:k+1])
feature_extractor.build((None, 30000, 1))
feature_extractor.trainable = False

test_out = feature_extractor(dummy)
print(f"Sortie : {test_out.shape}")   # must display (1, 64)

Sortie : (1, 64)


In [9]:
inputs = tf.keras.Input(shape=(30000, 1))
x = feature_extractor(inputs, training=False)
x = tf.keras.layers.Dense(32, activation='relu')(x)
x = tf.keras.layers.Dropout(0.3)(x)
output = tf.keras.layers.Dense(n_fonctions, activation='sigmoid', name='fonctions')(x)

model_ml = tf.keras.Model(inputs=inputs, outputs=output)
model_ml.summary()

Model: "functional_15"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 30000, 1)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 64)             │       167,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fonctions (Dense)               │ (None, 15)             │           495 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 170,287 (665.18 KB)

 Trainable params: 2,575 (10.06 KB)

 Non-trainable params: 167,712 (655.12 KB)

In [ ]:
def weighted_bce(pos_weights):
    w = tf.constant(pos_weights, dtype=tf.float32)
    def loss(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        bce = -(w * y_true * tf.math.log(y_pred)
                + (1 - y_true) * tf.math.log(1 - y_pred))
        return tf.reduce_mean(bce)
    return loss

model_ml.compile(
    optimizer='adam',
    loss=weighted_bce(pos_weights),
    metrics=[tf.keras.metrics.AUC(name='auc', multi_label=True)]
)

gen_ml = SpectreGeneratorMultiLabel(
    molecules_1H, df_annotations, colonnes_fonctions,
    points_1H, batch_size=128, n_per_molecule=50, augmentation=True
)

callbacks_ml = [
    tf.keras.callbacks.EarlyStopping(monitor='val_auc', mode='max',
                                     patience=7, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(
        '/content/drive/MyDrive/IA RMN/mon_modele_tf 1H/modele_prediction_RMN1H.keras',
        monitor='val_auc', mode='max', save_best_only=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3)
]

history_ml = model_ml.fit(
    gen_ml, epochs=30,
    validation_data=(X_val_ml, Y_val_ml),
    callbacks=callbacks_ml
)

Molécules avec annotations : 1231
Epoch 1/30
480/480 ━━━━━━━━━━━━━━━━━━━━ 59s 109ms/step - auc: 0.5838 - loss: 0.9416 - val_auc: 0.7275 - val_loss: 0.6269 - learning_rate: 0.0010
Epoch 2/30
480/480 ━━━━━━━━━━━━━━━━━━━━ 52s 107ms/step - auc: 0.7368 - loss: 0.6122 - val_auc: 0.8194 - val_loss: 0.5374 - learning_rate: 0.0010
Epoch 3/30
480/480 ━━━━━━━━━━━━━━━━━━━━ 49s 102ms/step - auc: 0.7998 - loss: 0.5507 - val_auc: 0.8602 - val_loss: 0.4782 - learning_rate: 0.0010
Epoch 4/30
480/480 ━━━━━━━━━━━━━━━━━━━━ 51s 106ms/step - auc: 0.8323 - loss: 0.5100 - val_auc: 0.8793 - val_loss: 0.4460 - learning_rate: 0.0010
Epoch 5/30
480/480 ━━━━━━━━━━━━━━━━━━━━ 50s 102ms/step - auc: 0.8503 - loss: 0.4842 - val_auc: 0.8918 - val_loss: 0.4231 - learning_rate: 0.0010
Epoch 6/30
480/480 ━━━━━━━━━━━━━━━━━━━━ 50s 102ms/step - auc: 0.8631 - loss: 0.4649 - val_auc: 0.9022 - val_loss: 0.4045 - learning_rate: 0.0010
Epoch 7/30
480/480 ━━━━━━━━━━━━━━━━━━━━ 50s 102ms/step - auc: 0.8720 - loss: 0.4505 - val_auc: 0

In [ ]:
import os

dossier_sauvegarde = '/content/drive/MyDrive/IA RMN/mon_modele_tf 1H'

if not os.path.exists(dossier_sauvegarde):
    os.makedirs(dossier_sauvegarde)

chemin_h5 = os.path.join(dossier_sauvegarde, 'modele_prediction_RMN1H.h5')
model_ml.save(chemin_h5)

In [ ]:
from sklearn.metrics import f1_score

Y_pred_proba = model_ml.predict(X_val_ml, batch_size=128, verbose=1)

print("Seuil optimal par fonction :")
seuils_optimaux = {}
for i, fonction in enumerate(colonnes_fonctions):
    best_f1, best_t = 0, 0.5
    for t in np.arange(0.1, 0.91, 0.05):
        f = f1_score(Y_val_ml[:, i], (Y_pred_proba[:, i] >= t).astype(int), zero_division=0)
        if f > best_f1:
            best_f1, best_t = f, t
    seuils_optimaux[fonction] = best_t
    print(f"  {fonction:<22} seuil={best_t:.2f}  F1={best_f1:.3f}")

Y_pred_opt = np.zeros_like(Y_pred_proba, dtype=int)
for i, fonction in enumerate(colonnes_fonctions):
    Y_pred_opt[:, i] = (Y_pred_proba[:, i] >= seuils_optimaux[fonction]).astype(int)
print(f"\nF1 macro : {f1_score(Y_val_ml, Y_pred_opt, average='macro', zero_division=0):.3f}")

96/96 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step
Seuil optimal par fonction :
  aromatique             seuil=0.45  F1=0.915
  alcool                 seuil=0.45  F1=0.768
  phenol                 seuil=0.55  F1=0.749
  acide_carboxylique     seuil=0.45  F1=0.652
  cetone                 seuil=0.50  F1=0.688
  aldehyde               seuil=0.45  F1=0.903
  amine                  seuil=0.45  F1=0.630
  ester                  seuil=0.45  F1=0.682
  ether                  seuil=0.50  F1=0.860
  halogenure             seuil=0.50  F1=0.568
  nitrile                seuil=0.50  F1=0.673
  amide                  seuil=0.40  F1=0.545
  heterocycle_n          seuil=0.45  F1=0.758
  sulfoxyde              seuil=0.55  F1=0.560
  alcene                 seuil=0.55  F1=0.776

F1 macro avec seuils optimisés : 0.715


In [ ]:
with open('/content/drive/MyDrive/IA RMN/mon_modele_tf 1H/seuils_optimaux_1H.json', 'w') as f:
    json.dump(seuils_optimaux, f, indent=2)

In [11]:
# Load the model
model_prediction = tf.keras.models.load_model(
    '/content/drive/MyDrive/IA RMN/mon_modele_tf 1H/modele_prediction_RMN1H.keras',
    compile=False
)

# Load the optimal thresholds
with open('/content/drive/MyDrive/IA RMN/mon_modele_tf 1H/seuils_optimaux_1H.json') as f:
    seuils_optimaux = json.load(f)

In [28]:
def predire_fonctions(csv_path, n_essais=10):
    df = pd.read_csv(csv_path)[['ppm', 'Int.']]
    df['ppm'] = pd.to_numeric(df['ppm'], errors='coerce')
    df['Int.'] = pd.to_numeric(df['Int.'], errors='coerce')
    df.dropna(inplace=True)
    df = normaliser_intensites(df)

    ppm_arr = df['ppm'].values
    int_arr = df['Int.'].values

    preds = []
    for _ in range(n_essais):
        X_rand  = np.random.uniform(-0.1, 0.1)
        Y_rand  = np.random.uniform(0.8, 1.2)
        largeur = np.random.uniform(0.0015, 0.004)

        spectre = np.zeros_like(points_1H)
        for p, h in zip(ppm_arr, int_arr):
            spectre += crpic(points_1H, p + X_rand, h * Y_rand, largeur)

        amax = np.max(spectre)
        snr = 10 ** np.random.uniform(np.log10(150), np.log10(500))
        spectre += np.random.normal(0, amax / snr, len(points_1H))

        x = spectre[np.newaxis, ..., np.newaxis].astype(np.float32)
        preds.append(model_prediction(x, training=False).numpy()[0])

    predictions = np.mean(preds, axis=0)

    print(f"Fonctions détectées ({n_essais} tirages moyennés) :")
    print("-" * 55)
    detectees = []
    for fonction, proba in zip(colonnes_fonctions, predictions):
        seuil = seuils_optimaux[fonction]
        marque = "OUI" if proba >= seuil else "non"
        print(f"  {fonction:<22} {proba*100:5.1f}%  (seuil {seuil:.2f})  {marque}")
        if proba >= seuil:
            detectees.append(fonction)

    print(f"\n→ Fonctions présentes : {', '.join(detectees) if detectees else 'aucune'}")
    return predictions

In [31]:
predire_fonctions('/content/drive/MyDrive/IA RMN/Molecules CSV 1H test/Acetone.csv')

Fonctions détectées (10 tirages moyennés) :
-------------------------------------------------------
  aromatique               2.0%  (seuil 0.45)  non
  alcool                  12.1%  (seuil 0.45)  non
  phenol                   0.1%  (seuil 0.55)  non
  acide_carboxylique      49.2%  (seuil 0.45)  OUI
  cetone                  56.7%  (seuil 0.50)  OUI
  aldehyde                 0.0%  (seuil 0.45)  non
  amine                   14.4%  (seuil 0.45)  non
  ester                   24.7%  (seuil 0.45)  non
  ether                    0.1%  (seuil 0.50)  non
  halogenure               8.6%  (seuil 0.50)  non
  nitrile                  7.4%  (seuil 0.50)  non
  amide                   19.8%  (seuil 0.40)  non
  heterocycle_n            2.2%  (seuil 0.45)  non
  sulfoxyde                0.7%  (seuil 0.55)  non
  alcene                   1.4%  (seuil 0.55)  non

→ Fonctions présentes : acide_carboxylique, cetone


array([2.0390455e-02, 1.2054678e-01, 5.3863839e-04, 4.9155051e-01,
       5.6701130e-01, 1.5293703e-04, 1.4415859e-01, 2.4685542e-01,
       1.0666292e-03, 8.5838690e-02, 7.4490838e-02, 1.9823900e-01,
       2.2484003e-02, 7.3403185e-03, 1.4360110e-02], dtype=float32)